In [ ]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("Proyecto", 
                "spark://spark-master:7077")
spark = su.spark

spark.conf.set("spark.sql.shuffle.partitions", "50")
print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

In [6]:
from pyspark.sql.functions import col, when, percentile_approx

In [7]:
campaigns_schema = SparkUtils.generate_schema(
    [
        ("campaign_id", "string"),
        ("campaign_name", "string"),
        ("advertiser_name", "string"),
        ("campaign_type", "string"),
        ("ad_category", "string"),
        ("target_country", "string"),
        ("budget_usd", "double"),
        ("start_date", "date"),
        ("end_date", "date"),
        ("is_active", "int"),
    ]
)

campaigns_raw= (
    spark.read.schema(campaigns_schema)
    .option("sep", "\t")
    .option("header", "true")
    .option("dateFormat", "yyyy-MM-dd")
    .csv("/opt/spark/work-dir/data/proyecto/campaigns/")
)

In [8]:
impressions_schema = SparkUtils.generate_schema(
    [
        ("impression_id", "string"),
        ("timestamp", "timestamp"),
        ("day", "int"),
        ("user_id", "string"),
        ("session_id", "string"),
        ("device_type", "string"),
        ("os", "string"),
        ("browser", "string"),
        ("country", "string"),
        ("region", "string"),
        ("city", "string"),
        ("ad_id", "string"),
        ("campaign_id", "string"),
        ("advertiser_id", "string"),
        ("ad_category", "string"),
        ("ad_position", "string"),
        ("ad_format", "string"),
        ("page_url_hash", "string"),
        ("referrer_hash", "string"),
        ("user_age_bucket", "string"),
        ("user_gender", "string"),
        ("user_interest_1", "string"),
        ("user_interest_2", "string"),
        ("page_views_today", "int"),
        ("time_on_site_sec", "int"),
        ("recency_days", "int"),
        ("frequency_30d", "int"),
        ("bid_price_usd", "double"),
        ("floor_price_usd", "double"),
        ("quality_score", "double"),
        ("predicted_ctr", "double"),
        ("hour_of_day", "int"),
        ("day_of_week", "int"),
        ("is_weekend", "int"),
        ("label", "int"),
    ]
)

impressions_raw = (
    spark.read.schema(impressions_schema)
    .option("sep", "\t")
    .option("header", "true")
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .csv("/opt/spark/work-dir/data/proyecto/days/")
)

In [ ]:
LOW = 0.01
HIGH = 0.99

imp_percentiles = impressions_raw.select(
    percentile_approx("bid_price_usd",   [LOW, HIGH]).alias("bid_p"),
    percentile_approx("floor_price_usd", [LOW, HIGH]).alias("floor_p"),
    percentile_approx("quality_score",   [LOW, HIGH]).alias("qs_p"),
    percentile_approx("predicted_ctr",   [LOW, HIGH]).alias("ctr_p"),
).first()

camp_percentiles = campaigns_raw.select(
    percentile_approx("budget_usd", [LOW, HIGH]).alias("budget_p"),
).first()

In [10]:
class Percentiles:
    def __init__ (self, arr):
        self._01 = arr[0]
        self._99 = arr[1]

In [11]:
bid_price_usd_percentiles = Percentiles(imp_percentiles['bid_p'])
floor_price_usd_percentiles = Percentiles(imp_percentiles['floor_p'])
quality_score_percentiles = Percentiles(imp_percentiles['qs_p'])
predicted_ctr_percentiles = Percentiles(imp_percentiles['ctr_p'])
budget_usd_percentiles = Percentiles(camp_percentiles['budget_p'])

In [14]:
IMPRESSIONS_COLS = [
    "impression_id",
    "day",
    "campaign_id",
    "ad_category",
    "day",
    "country",
    "device_type",
    "bid_price_usd",
    "floor_price_usd",
    "quality_score",
    "predicted_ctr",
    "label",
    "is_weekend",
    
]

CAMPAIGNS_COLS = [
    "campaign_id",
    "advertiser_name",
    "campaign_type",
    "budget_usd",
    "is_active",
]

In [ ]:

impressions_clean = (
    impressions_raw
    .select(IMPRESSIONS_COLS)
    .dropna()
    .dropDuplicates(["impression_id"])
    .filter(col("bid_price_usd")  .between(bid_price_usd_percentiles._01,   bid_price_usd_percentiles._99))
    .filter(col("floor_price_usd").between(floor_price_usd_percentiles._01, floor_price_usd_percentiles._99))
    .filter(col("quality_score")  .between(quality_score_percentiles._01,    quality_score_percentiles._99))
    .filter(col("predicted_ctr")  .between(predicted_ctr_percentiles._01,   predicted_ctr_percentiles._99))
    .filter(col("bid_price_usd")  >= col("floor_price_usd"))
)

campaigns_clean = (
    campaigns_raw
    .select(CAMPAIGNS_COLS)
    .dropDuplicates(["campaign_id"])
    .fillna({"is_active": 0})
    .dropna()
    .filter(col("budget_usd").between(budget_usd_percentiles._01, budget_usd_percentiles._99))
)


In [ ]:
cols = set(IMPRESSIONS_COLS + CAMPAIGNS_COLS)
cols = list(cols)

df = (
    impressions_clean
    .join(campaigns_clean, on="campaign_id", how="left")
    .select(cols)
    .withColumn("revenue",
        col("bid_price_usd") * col("label"))
    .withColumn("bid_margin",
        col("bid_price_usd") - col("floor_price_usd"))
    .withColumn("ctr_error",
        col("predicted_ctr") - col("label"))
    .withColumn("ctr_bucket",
        when(col("predicted_ctr") < 0.05,  "low")
        .when(col("predicted_ctr") < 0.15, "medium")
        .otherwise("high"))
    .withColumn("quality_bucket",
        when(col("quality_score") < 3.0,  "low")
        .when(col("quality_score") < 7.0, "medium")
        .otherwise("high"))
)

In [17]:
df.show()

+--------------------+---+-------------+-----------+----------+-------------+---------+-------------+-------------+----------+-------+--------------------+---------------+-------------+-----------+-----+-------+--------------------+-------------------+---------+----------+--------------+
|     advertiser_name|day|  ad_category|device_type|budget_usd|predicted_ctr|is_active|quality_score|bid_price_usd|is_weekend|country|       impression_id|floor_price_usd|campaign_type|campaign_id|label|revenue|                 roi|         bid_margin|ctr_error|ctr_bucket|quality_bucket|
+--------------------+---+-------------+-----------+----------+-------------+---------+-------------+-------------+----------+-------+--------------------+---------------+-------------+-----------+-----+-------+--------------------+-------------------+---------+----------+--------------+
|         Shepard PLC|  8|  real_estate|    desktop| 436135.17|     0.026427|        0|        4.364|        1.674|         0|     CO

In [23]:
df.write \
    .mode("overwrite") \
    .partitionBy("day") \
    .parquet("/opt/spark/work-dir/data/proyecto/output/clean_data/")